# Test a mesh-transformer checkpoint

Loads the **same** test split a training run used -- same config, same seed, same
`random_split` -- generates LOD2 from each LOD1 condition, and shows
input / ground truth / generated side by side.

Generation has no KV cache: ~9 forward passes per triangle, so keep `N_SHOW` small.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch

from src.dataset.mesh_datamodule import MeshDataModule
from src.dataset.mesh_dataset import detokenize, mesh_collate_fn
from src.eval.mesh_metrics import mesh_metrics
from src.models.mesh_transformer import MeshTransformerModule
from src.utils.initialization import load_config
from src.visualize_mesh import cityjson_figure_from_mesh, mesh_figure, side_by_side

## The experiment to test

`CKPT = None` picks the newest checkpoint of the run the config names.

In [ ]:
CONFIG = ROOT / "configs" / "mesh-train.yaml"
CKPT = None          # or a path to a specific .ckpt
N_SHOW = 4           # buildings to generate; each one is slow
TEMPERATURE = 0.0    # 0 = argmax, as in mesh_eval

cfg = load_config(CONFIG, [])
run_dir = ROOT / cfg.logging.save_dir / cfg.logging.experiment_name / cfg.logging.run_name

if CKPT is None:
    ckpts = sorted(run_dir.rglob("*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        raise FileNotFoundError(f"No .ckpt under {run_dir} -- set CKPT explicitly.")
    CKPT = ckpts[-1]

print(f"config {CONFIG.name}, run {cfg.logging.run_name}")
print(f"checkpoint {Path(CKPT).relative_to(ROOT)}")

## The same test split

Every argument below is what `src/train_mesh.py` passes, `cfg.seed` included, so
`random_split` reproduces the exact partition the run trained against.

In [ ]:
datamodule = MeshDataModule(
    dataset_dir=ROOT / cfg.mesh_data.dataset_dir,
    lod_in=cfg.mesh_data.lod_in,
    lod_out=cfg.mesh_data.lod_out,
    num_bins=cfg.mesh_data.num_bins,
    margin_lo=list(cfg.mesh_data.margin_lo),
    margin_hi=list(cfg.mesh_data.margin_hi),
    max_faces=cfg.mesh_data.max_faces,
    max_files=cfg.mesh_data.max_files,
    batch_size=cfg.training.batch_size,
    train_val_test_split=tuple(cfg.training.train_val_test_split),
    num_workers=0,
    seed=cfg.seed,
)
datamodule.setup()
test_set = datamodule.test_dataset

print(f"train={len(datamodule.train_dataset)} val={len(datamodule.val_dataset)} "
      f"test={len(test_set)}")

In [ ]:
model = MeshTransformerModule.load_from_checkpoint(CKPT, map_location="cpu")
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Same budget rule as MeshEvalCallback: asking past the positional embedding raises.
max_new_tokens = min(9 * (cfg.mesh_data.max_faces or 200) + 1,
                     model.network.max_seq_len - 1)
print(f"{model.num_bins} bins, max_seq_len {model.network.max_seq_len}, "
      f"budget {max_new_tokens} tokens on {device}")

## Generate

In [ ]:
def to_metres(tokens, center, scale):
    """Token sequence -> (verts, faces) in metres, undoing the LOD1-box frame."""
    verts, faces = detokenize(tokens, model.num_bins)
    return verts * scale + center, faces


items = [test_set[i] for i in range(min(N_SHOW, len(test_set)))]
batch = mesh_collate_fn(items, pad=model.pad)

with torch.no_grad():
    out = model.generate(batch["cond"].to(device),
                         batch["cond_pad_mask"].to(device),
                         max_new_tokens=max_new_tokens,
                         temperature=TEMPERATURE)

results = []
for k, name in enumerate(batch["ids"]):
    center, scale = batch["center"][k].numpy(), batch["scale"][k].numpy()
    lod1 = to_metres(batch["cond"][k].numpy(), center, scale)
    gt = to_metres(batch["tgt"][k].numpy(), center, scale)
    gen = to_metres(out[k].cpu().numpy(), center, scale)
    results.append((name, lod1, gt, gen))
    print(f"{name}: lod1 {len(lod1[1])} tris | gt {len(gt[1])} | gen {len(gen[1])}")

## Input / ground truth / generated

Top row: the raw triangle meshes, all three after the tokenizer round trip, so what
is left between the middle and right panel is model error alone.

Bottom row: the same geometry through `mesh_to_cityjson` -- coplanar triangles merged
back into polygons and coloured by semantic surface (blue ground, orange roof, grey
wall). An empty bottom panel means the writer refused the mesh.

In [ ]:
from IPython.display import display

for name, lod1, gt, gen in results:
    row = mesh_metrics(gen, gt, taus=tuple(cfg.mesh_eval.taus),
                       n_points=cfg.mesh_eval.n_points, voxel_m=cfg.mesh_eval.voxel_m)
    print(name, {k: round(v, 4) for k, v in sorted(row.items()) if np.isfinite(v)})

    meshes = [(lod1, "#898781"), (gt, "#2a78d6"), (gen, "#eda100")]
    top = [mesh_figure(*mesh, color=color) for mesh, color in meshes]
    bottom = [cityjson_figure_from_mesh(*mesh) for mesh, _ in meshes]

    # display(), not fig.show(): show() picks a renderer and draws nothing when
    # it guesses wrong, while display() emits the same mime bundle a bare figure
    # at the end of a cell does -- which is what already renders elsewhere.
    display(side_by_side(
        [top, [fig for fig, _ in bottom]],
        [f"LOD1 input, {len(lod1[1])} tris",
         f"LOD2 ground truth, {len(gt[1])} tris",
         f"LOD2 generated, {len(gen[1])} tris"]
        + [f"CityJSON, {n} surfaces" if n else "CityJSON: none" for _, n in bottom],
    ))